# Regularização no Dataset Wine

Este notebook testa diferentes estratégias de regularização (L1, L2, Dropout) no dataset Wine do scikit-learn.

**Dataset:** Wine Recognition Dataset
- **Tipo:** Classificação multiclasse (3 tipos de vinho)
- **Features:** 13 características químicas
- **Amostras:** ~178 amostras


In [ ]:
import tensorflow as tf
from keras import layers, regularizers
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

print(f"TensorFlow version: {tf.__version__}")
print(f"Num GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")


In [ ]:
# Carregar e preparar o dataset Wine
data = load_wine()
X, y = data.data, data.target
feature_names = data.feature_names
target_names = data.target_names

print(f"Dataset shape: {X.shape}")
print(f"Features: {len(feature_names)}")
print(f"Classes: {target_names}")
print(f"Class distribution: {np.bincount(y)}")

# Normalizar features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Dividir em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Converter para tensores do TensorFlow
X_train = tf.constant(X_train, dtype=tf.float32)
X_test = tf.constant(X_test, dtype=tf.float32)
y_train = tf.constant(y_train, dtype=tf.int32)
y_test = tf.constant(y_test, dtype=tf.int32)

# Converter para one-hot encoding para classificação multiclasse
y_train = tf.one_hot(y_train, 3)
y_test = tf.one_hot(y_test, 3)

print(f"\nTrain set: {X_train.shape}, {y_train.shape}")
print(f"Test set: {X_test.shape}, {y_test.shape}")


In [ ]:
def create_model(model_type, n_features, n_classes=3, hidden_units=64, dropout_rate=0.5, l1_reg=0.001, l2_reg=0.001):
    """Cria diferentes tipos de modelos com regularização para classificação multiclasse"""
    
    model = tf.keras.Sequential()
    model.add(layers.Input(shape=(n_features,)))
    
    if model_type == 'baseline':
        # Modelo baseline sem regularização
        model.add(layers.Dense(hidden_units, activation='relu'))
        model.add(layers.Dense(hidden_units, activation='relu'))
        model.add(layers.Dense(hidden_units, activation='relu'))
        
    elif model_type == 'l1':
        # Modelo com regularização L1
        model.add(layers.Dense(hidden_units, activation='relu', 
                              kernel_regularizer=regularizers.l1(l1_reg)))
        model.add(layers.Dense(hidden_units, activation='relu', 
                              kernel_regularizer=regularizers.l1(l1_reg)))
        model.add(layers.Dense(hidden_units, activation='relu', 
                              kernel_regularizer=regularizers.l1(l1_reg)))
        
    elif model_type == 'l2':
        # Modelo com regularização L2
        model.add(layers.Dense(hidden_units, activation='relu', 
                              kernel_regularizer=regularizers.l2(l2_reg)))
        model.add(layers.Dense(hidden_units, activation='relu', 
                              kernel_regularizer=regularizers.l2(l2_reg)))
        model.add(layers.Dense(hidden_units, activation='relu', 
                              kernel_regularizer=regularizers.l2(l2_reg)))
        
    elif model_type == 'dropout':
        # Modelo com Dropout
        model.add(layers.Dense(hidden_units, activation='relu'))
        model.add(layers.Dropout(dropout_rate))
        model.add(layers.Dense(hidden_units, activation='relu'))
        model.add(layers.Dropout(dropout_rate))
        model.add(layers.Dense(hidden_units, activation='relu'))
        model.add(layers.Dropout(dropout_rate))
        
    elif model_type == 'combined':
        # Modelo com L2 + Dropout
        model.add(layers.Dense(hidden_units, activation='relu', 
                              kernel_regularizer=regularizers.l2(l2_reg)))
        model.add(layers.Dropout(dropout_rate))
        model.add(layers.Dense(hidden_units, activation='relu', 
                              kernel_regularizer=regularizers.l2(l2_reg)))
        model.add(layers.Dropout(dropout_rate))
        model.add(layers.Dense(hidden_units, activation='relu', 
                              kernel_regularizer=regularizers.l2(l2_reg)))
        model.add(layers.Dropout(dropout_rate))
    
    # Camada de saída para classificação multiclasse
    model.add(layers.Dense(n_classes, activation='softmax'))
    
    return model

print("Função de criação de modelos definida!")


In [ ]:
def train_model(model, X_train, y_train, X_test, y_test, epochs=100, batch_size=32, verbose=0):
    """Treina um modelo e retorna o histórico"""
    
    # Compilar modelo
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Callbacks
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=15, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7
        )
    ]
    
    # Treinar
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=verbose
    )
    
    return history

print("Função de treinamento definida!")


In [ ]:
# Experimento: Comparação de Regularização no Wine
model_types = ['baseline', 'l1', 'l2', 'dropout', 'combined']
results = {}

print("Iniciando experimentos de regularização no Wine...")
print("=" * 60)

for model_type in model_types:
    print(f"\nTreinando modelo: {model_type.upper()}")
    print("-" * 30)
    
    # Criar modelo
    model = create_model(
        model_type, 
        n_features=X_train.shape[1],
        n_classes=3,
        hidden_units=32,  # Menor para dataset pequeno
        dropout_rate=0.3, 
        l1_reg=0.01, 
        l2_reg=0.01
    )
    
    # Treinar modelo
    history = train_model(
        model, X_train, y_train, X_test, y_test,
        epochs=100, verbose=0
    )
    
    # Avaliar modelo
    train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    
    # Salvar resultados
    results[model_type] = {
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'train_loss': train_loss,
        'test_loss': test_loss,
        'history': history,
        'model': model
    }
    
    print(f"  Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")
    print(f"  Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")
    print(f"  Overfitting: {train_acc - test_acc:.4f}")

print("\nExperimentos concluídos!")


In [ ]:
# Análise dos Resultados
results_df = pd.DataFrame([
    {
        'Model': model_type,
        'Train_Accuracy': model_results['train_accuracy'],
        'Test_Accuracy': model_results['test_accuracy'],
        'Train_Loss': model_results['train_loss'],
        'Test_Loss': model_results['test_loss'],
        'Overfitting': model_results['train_accuracy'] - model_results['test_accuracy']
    }
    for model_type, model_results in results.items()
])

print("Resultados dos Experimentos - Wine:")
print("=" * 60)
print(results_df.round(4))

# Encontrar melhor modelo
best_model = results_df.loc[results_df['Overfitting'].idxmin()]
print(f"\nMelhor modelo (menor overfitting): {best_model['Model']}")
print(f"  Train Acc: {best_model['Train_Accuracy']:.4f}")
print(f"  Test Acc: {best_model['Test_Accuracy']:.4f}")
print(f"  Overfitting: {best_model['Overfitting']:.4f}")


In [ ]:
# Visualizações
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Comparação de Acurácia
ax1 = axes[0, 0]
models = list(results.keys())
train_accs = [results[model]['train_accuracy'] for model in models]
test_accs = [results[model]['test_accuracy'] for model in models]

x = np.arange(len(models))
width = 0.35
ax1.bar(x - width/2, train_accs, width, label='Train', alpha=0.8)
ax1.bar(x + width/2, test_accs, width, label='Test', alpha=0.8)
ax1.set_xlabel('Modelo')
ax1.set_ylabel('Acurácia')
ax1.set_title('Comparação de Acurácia - Wine')
ax1.set_xticks(x)
ax1.set_xticklabels(models, rotation=45)
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Overfitting
ax2 = axes[0, 1]
overfitting = [results[model]['train_accuracy'] - results[model]['test_accuracy'] for model in models]
ax2.bar(models, overfitting, alpha=0.8, color='red')
ax2.set_xlabel('Modelo')
ax2.set_ylabel('Overfitting (Train - Test)')
ax2.set_title('Overfitting por Modelo')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True, alpha=0.3)

# 3. Curvas de Loss
ax3 = axes[1, 0]
for model_type in models:
    history = results[model_type]['history']
    ax3.plot(history.history['loss'], label=f'{model_type} (train)')
    ax3.plot(history.history['val_loss'], label=f'{model_type} (val)', linestyle='--')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Loss')
ax3.set_title('Curvas de Loss')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Curvas de Accuracy
ax4 = axes[1, 1]
for model_type in models:
    history = results[model_type]['history']
    ax4.plot(history.history['accuracy'], label=f'{model_type} (train)')
    ax4.plot(history.history['val_accuracy'], label=f'{model_type} (val)', linestyle='--')
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Accuracy')
ax4.set_title('Curvas de Accuracy')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
